# NM i AI 2026 — NorgesGruppen Object Detection (Kaggle v5)
**YOLOv8x | imgsz=1280 | T4 x2**

**Setup checklist:**
1. *Settings → Accelerator → GPU T4 x2*
2. *Add data → nm-ai-2026-dataset*
3. Run **Cell 1** (writes the script), then **Cell 2** (runs it)

Download `best_grocery_nm2026.pt` from the *Output* tab when done.

In [ ]:
%%writefile /kaggle/working/train.py
import subprocess, sys, json, random, shutil, os, functools
from pathlib import Path

# Install ultralytics into a private folder so it loads before
# Kaggle's broken system version
TARGET = "/kaggle/working/ultralytics_pkg"
subprocess.run([sys.executable, "-m", "pip", "install",
                "ultralytics", "--target", TARGET, "-q"], check=True)
sys.path.insert(0, TARGET)

# Also set PYTHONPATH so DDP subprocesses (which spawn fresh Python)
# can find our ultralytics installation too
os.environ["PYTHONPATH"] = TARGET + ":" + os.environ.get("PYTHONPATH", "")

# PyTorch 2.6 fix
import torch
torch.load = functools.partial(torch.load, weights_only=False)

from ultralytics import YOLO
import ultralytics
print(f"PyTorch:     {torch.__version__}")
print(f"Ultralytics: {ultralytics.__version__}")
print(f"Loaded from: {ultralytics.__file__}")
for i in range(torch.cuda.device_count()):
    vram = torch.cuda.get_device_properties(i).total_memory / 1e9
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}  ({vram:.1f} GB)")

# ── Paths ──────────────────────────────────────────────────────────
WORKING_DIR = Path("/kaggle/working")
matches = list(Path("/kaggle/input").rglob("annotations.json"))
if not matches:
    raise FileNotFoundError("annotations.json not found — did you add the dataset?")
ANNOTATIONS_FILE = matches[0]
COCO_TRAIN_DIR   = ANNOTATIONS_FILE.parent
IMAGES_DIR       = COCO_TRAIN_DIR / "images"
n_imgs = len(list(IMAGES_DIR.glob("*.jpg"))) + len(list(IMAGES_DIR.glob("*.jpeg")))
print(f"Dataset: {COCO_TRAIN_DIR}")
print(f"Images:  {n_imgs}")
if n_imgs == 0:
    raise FileNotFoundError(f"No images found in {IMAGES_DIR}")

# ── COCO -> YOLO labels ────────────────────────────────────────────
LABELS_DIR = WORKING_DIR / "labels"
LABELS_DIR.mkdir(parents=True, exist_ok=True)
with open(ANNOTATIONS_FILE, encoding="utf-8") as f:
    coco = json.load(f)
id_to_img  = {img["id"]: img for img in coco["images"]}
id_to_anns = {}
for ann in coco["annotations"]:
    id_to_anns.setdefault(ann["image_id"], []).append(ann)
for img_id, img_info in id_to_img.items():
    w, h = img_info["width"], img_info["height"]
    stem = Path(img_info["file_name"]).stem
    lines = []
    for ann in id_to_anns.get(img_id, []):
        if ann.get("iscrowd", 0):
            continue
        x, y, bw, bh = ann["bbox"]
        xc = max(0.0, min(1.0, (x + bw/2) / w))
        yc = max(0.0, min(1.0, (y + bh/2) / h))
        wn = max(0.0, min(1.0, bw / w))
        hn = max(0.0, min(1.0, bh / h))
        lines.append(f"{ann['category_id']} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}")
    (LABELS_DIR / f"{stem}.txt").write_text("\n".join(lines), encoding="utf-8")
print(f"Written {len(id_to_img)} label files to {LABELS_DIR}")

# ── Symlink images so YOLO finds labels correctly ──────────────────
sym_images = WORKING_DIR / "images"
if not sym_images.exists():
    os.symlink(str(IMAGES_DIR), str(sym_images))
print(f"Images symlinked: {sym_images} -> {IMAGES_DIR}")

# ── Train/val split + YAML ─────────────────────────────────────────
YAML_PATH  = WORKING_DIR / "grocery_nm2026.yaml"
all_images = sorted(IMAGES_DIR.glob("*.jpg")) + sorted(IMAGES_DIR.glob("*.jpeg"))
random.seed(42)
random.shuffle(all_images)
n_val   = max(1, round(len(all_images) * 0.10))
val_set = all_images[:n_val]
trn_set = all_images[n_val:]
(WORKING_DIR / "train.txt").write_text(
    "\n".join(str(sym_images / p.name) for p in trn_set), encoding="utf-8")
(WORKING_DIR / "val.txt").write_text(
    "\n".join(str(sym_images / p.name) for p in val_set), encoding="utf-8")
print(f"Split: {len(trn_set)} train / {len(val_set)} val")
categories = sorted(coco["categories"], key=lambda c: c["id"])
nc    = len(categories)
names = [c["name"].replace("'", "") for c in categories]
YAML_PATH.write_text(
    f"path: {WORKING_DIR.as_posix()}\n"
    f"train: train.txt\n"
    f"val:   val.txt\n\n"
    f"nc: {nc}\n"
    f"names:\n" + "\n".join(f"  - '{n}'" for n in names) + "\n",
    encoding="utf-8"
)
print(f"YAML written ({nc} classes)")

# ── Callback: copy best.pt every 5 epochs during training ─────────
CHECKPOINT = WORKING_DIR / "best_grocery_nm2026.pt"

def save_best_every_5(trainer):
    if (trainer.epoch + 1) % 5 == 0:
        best_src = Path(trainer.save_dir) / "weights" / "best.pt"
        if best_src.exists():
            shutil.copy(best_src, CHECKPOINT)
            print(f"  [checkpoint] epoch {trainer.epoch + 1} → {CHECKPOINT} "
                  f"({CHECKPOINT.stat().st_size / 1e6:.1f} MB)")

# ── Train ──────────────────────────────────────────────────────────
model = YOLO("yolov8x.pt")
model.add_callback("on_fit_epoch_end", save_best_every_5)
results = model.train(
    data         = str(YAML_PATH),
    epochs       = 150,
    imgsz        = 1280,
    batch        = 8,
    device       = "0,1",
    workers      = 2,
    cache        = False,
    project      = str(WORKING_DIR / "runs"),
    name         = "grocery_nm2026",
    exist_ok     = True,
    augment      = True,
    mosaic       = 1.0,
    mixup        = 0.15,
    copy_paste   = 0.3,
    fliplr       = 0.5,
    hsv_h        = 0.015,
    hsv_s        = 0.7,
    hsv_v        = 0.4,
    lr0          = 0.01,
    lrf          = 0.01,
    momentum     = 0.937,
    weight_decay = 0.0005,
    patience     = 50,
    close_mosaic = 15,
    save_period  = -1,
)

# Final save — always runs, even if results.box is None
best_src = WORKING_DIR / "runs" / "grocery_nm2026" / "weights" / "best.pt"
if best_src.exists():
    shutil.copy(best_src, CHECKPOINT)
    print(f"Final save: {CHECKPOINT}  ({CHECKPOINT.stat().st_size / 1e6:.1f} MB)")
else:
    print(f"WARNING: could not find {best_src}")

In [ ]:
# Cell 2 — run in a fresh subprocess, completely isolated from Kaggle's kernel
!python /kaggle/working/train.py